In [ ]:
import os
import re
import sys
import time
from pathlib import Path
import requests
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# Define target VN30 tickers
VN30_TICKERS = [
    "ACB", "BCM", "BID", "BVH", "CTG", "FPT", "GAS", "GVR", "HDB", "HPG",
    "MBB", "MSN", "MWG", "PLX", "POW", "SAB", "SHB", "SSB", "SSI", "STB",
    "TCB", "TPB", "VCB", "VHM", "VIB", "VIC", "VJC", "VNM", "VPB", "VRE"
]

# Output directory (data/raw inside project folder)
BASE_DATA_DIR = Path("..") / "data" / "raw"
print(f"Target directory: {BASE_DATA_DIR.resolve()}")

In [ ]:
def clean_filename(filename: str) -> str:
    return re.sub(r'[\\/*?:"<>|]', '', filename)

def parse_quarter_and_year(title: str):
    title_lower = title.lower()
    year_match = re.search(r'\b(202[0-9])\b', title_lower)
    if not year_match:
        return None, None
    year = int(year_match.group(1))
    
    quarter = None
    if 'quý 1' in title_lower or 'quý i' in title_lower or 'q1' in title_lower:
        quarter = '1'
    elif 'quý 2' in title_lower or 'quý ii' in title_lower or 'q2' in title_lower:
        quarter = '2'
    elif 'quý 3' in title_lower or 'quý iii' in title_lower or 'q3' in title_lower:
        quarter = '3'
    elif 'quý 4' in title_lower or 'quý iv' in title_lower or 'q4' in title_lower:
        quarter = '4'
    elif 'năm' in title_lower or 'cả năm' in title_lower or 'kiểm toán' in title_lower:
        quarter = 'Nam'
        
    return quarter, year

def download_file(url: str, save_path: Path, session_cookies=None):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    }
    session = requests.Session()
    if session_cookies:
        for cookie in session_cookies:
            session.cookies.set(cookie['name'], cookie['value'])
            
    try:
        response = session.get(url, headers=headers, stream=True, timeout=45)
        if response.status_code == 200:
            with open(save_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
            print(f'    Successfully downloaded: {save_path.name}')
            return True
        else:
            print(f'    Failed (Status {response.status_code}): {url}')
            return False
    except Exception as e:
        print(f'    Error downloading {url}: {e}')
        return False

In [ ]:
def scrape_vietstock_direct(symbol: str, target_years=[2023, 2024, 2025]):
    print(f'\n[{symbol}] Opening Vietstock corporate documents page via Headless Chrome...')
    
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')
    options.add_argument('--disable-gpu')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0')
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    
    try:
        url = f'https://finance.vietstock.vn/{symbol}/tai-lieu-cong-bo.htm'
        driver.get(url)
        
        # Wait for table contents to load
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.XPATH, "//a[contains(@href, '.pdf') or contains(@href, 'download') or contains(@href, 'Download')] | //table"))
        )
        time.sleep(3)
        
        links = driver.find_elements(By.TAG_NAME, 'a')
        candidates = []
        
        for l in links:
            try:
                href = l.get_attribute('href')
                title = l.text or l.get_attribute('title')
                if href and title:
                    href = href.strip()
                    title = title.strip()
                    is_bctc = any(kw in title.lower() for kw in ['báo cáo tài chính', 'bctc', 'báo cáo tài chính hợp nhất', 'bctc hợp nhất'])
                    is_pdf_link = href.lower().endswith('.pdf') or 'download' in href.lower() or 'Download' in href.lower() or 'file1.vietstock.vn' in href
                    
                    if is_bctc and is_pdf_link:
                        candidates.append((title, href))
            except Exception:
                continue
                
        candidates = list(set(candidates))
        print(f'  Found {len(candidates)} financial document link candidates.')
        
        cookies = driver.get_cookies()
        count = 0
        for title, pdf_url in candidates:
            quarter, year = parse_quarter_and_year(title)
            if not quarter or not year or year not in target_years:
                continue
                
            if not any(kw in title.lower() for kw in ['hợp nhất', 'hop nhat']):
                continue
                
            dest_dir = BASE_DATA_DIR / symbol / str(year)
            dest_dir.mkdir(parents=True, exist_ok=True)
            
            std_name = f'{symbol}_Baocaotaichinh_Q{quarter}_{year}_Hopnhat.pdf'
            save_path = dest_dir / std_name
            
            if save_path.exists():
                print(f'  File already exists: {std_name} (Skipping)')
                continue
                
            print(f'  -> Downloading: {title}')
            success = download_file(pdf_url, save_path, session_cookies=cookies)
            if success:
                count += 1
                time.sleep(2.0)
                
        print(f'[{symbol}] Downloaded {count} new reports.')
    except Exception as e:
        print(f'Error scraping {symbol} from Vietstock: {e}')
    finally:
        driver.quit()

In [ ]:
# Run test for VCB and ACB
for ticker in ['VCB', 'ACB']:
    scrape_vietstock_direct(ticker, target_years=[2023, 2024, 2025])
    time.sleep(3.0)